## AI was used as a coding assistant and markdown assistant. Not for idea generation.

In [3]:
SMOKE = False
DATA_DIR = './data/'
import numpy as np, pandas as pd, scipy.sparse as sp, lightgbm as lgb, catboost as cb, xgboost as xgb, warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from scipy.stats import rankdata
warnings.filterwarnings('ignore')
N_EST = 100 if SMOKE else 600
BAGS_A = (0,) if SMOKE else (0,11,22,33,44)
BAGS_B = (0,) if SMOKE else (0,11,22)
SEEDS = range(1) if SMOKE else range(3)
CAT_ITERS, CAT_SEEDS = (100, range(1)) if SMOKE else (1200, range(2))

## 1. Load data
`train.csv` holds time steps 1–35 (unlabelled rows have an empty `label`); `test.csv` holds time steps 36–49.

In [4]:
tr = pd.read_csv(DATA_DIR+'train.csv')
te = pd.read_csv(DATA_DIR+'test.csv'); edges = pd.read_csv(DATA_DIR+'txs_edgelist.csv')
te = te.rename(columns={'index':'test_index'}); tr['test_index'] = -1; te['label'] = np.nan
D = pd.concat([tr,te], ignore_index=True); D['is_test'] = (D.test_index>=0).astype(int)
RAW = [f'feat_{i}' for i in range(1,166)]
print('train', tr.shape, '| test', te.shape, '| edges', edges.shape)
print('labels', tr.label.value_counts(dropna=False).to_dict())
print('train ts', tr.time_step.min(), '-', tr.time_step.max(), '| test ts', te.time_step.min(), '-', te.time_step.max())

train (141772, 169) | test (15329, 169) | edges (234355, 2)
labels {nan: 110537, 0.0: 27591, 1.0: 3644}
train ts 1 - 35 | test ts 36 - 49


## 2. Data understanding
- Test = future time steps 36–49, and it contains **only labelled** transactions. The unlabelled ones were removed, but their edges remain in the edge list.
- Edges connect transactions within the same time step only.
- Illicit transactions cluster together.

In [5]:
ts = D.set_index('txId').time_step; lab = D.set_index('txId').label
e = edges.assign(t1=edges.txId1.map(ts), t2=edges.txId2.map(ts), l1=edges.txId1.map(lab), l2=edges.txId2.map(lab))
both = e.dropna(subset=['t1','t2'])
print('edges with both endpoints present:', len(both), '| same time step:', (both.t1==both.t2).mean())
print('edge endpoints missing (removed unlabelled test-period nodes):', int(e.t1.isna().sum()+e.t2.isna().sum()))
print(pd.crosstab(e.dropna(subset=['l1','l2']).l1, e.dropna(subset=['l1','l2']).l2, rownames=['src label'], colnames=['dst label']))

edges with both endpoints present: 175918 | same time step: 1.0
edge endpoints missing (removed unlabelled test-period nodes): 92629
dst label    0.0  1.0
src label            
0.0        21772  597
1.0          674  857


## 3. Graph feature engineering
- **Visible nodes** are labelled train nodes plus test nodes. All neighbour aggregates use visible nodes only. In the test period only labelled nodes have features, so this keeps train and test consistent.
- **Degrees** come from the full edge list: in, out, visible and hidden (unlabelled) neighbour counts, and 2-hop degree sums.
- **Neighbour aggregates** are computed for the 30 most predictive local features (univariate AUC): mean and max over visible predecessors and successors, own-minus-neighbour-mean differences, and the 2-hop mean.

In [6]:
L = D[D.label.notna()]
au = {f: abs(roc_auc_score(L.label, L[f])-.5) for f in [f'feat_{i}' for i in range(1,94)]}
TOP = sorted(au, key=au.get, reverse=True)[:30]
ids = pd.Index(pd.unique(np.concatenate([D.txId.values, edges.txId1.values, edges.txId2.values])))
pmap = pd.Series(np.arange(len(ids)), index=ids); n_all = len(ids)
A = sp.csr_matrix((np.ones(len(edges)), (pmap[edges.txId1].values, pmap[edges.txId2].values)), shape=(n_all,n_all))
node = pmap[D.txId].values
vis_all = np.zeros(n_all); vis_all[node[(D.label.notna()|(D.is_test==1)).values]] = 1
G = {}
indeg = np.asarray(A.sum(0)).ravel(); outdeg = np.asarray(A.sum(1)).ravel()
G['in_deg']=indeg[node]; G['out_deg']=outdeg[node]; G['deg']=G['in_deg']+G['out_deg']
G['in_vis']=(A.T@vis_all)[node]; G['out_vis']=(A@vis_all)[node]
G['in_hid']=G['in_deg']-G['in_vis']; G['out_hid']=G['out_deg']-G['out_vis']
with np.errstate(divide='ignore', invalid='ignore'): G['frac_vis']=np.where(G['deg']>0,(G['in_vis']+G['out_vis'])/G['deg'],np.nan)
G['in2']=(A.T@indeg)[node]; G['out2']=(A@outdeg)[node]
G['nb_in_outdeg_mean']=(A.T@outdeg)[node]/np.maximum(G['in_deg'],1)
G['nb_out_indeg_mean']=(A@indeg)[node]/np.maximum(G['out_deg'],1)
M = np.zeros((n_all,len(TOP)),dtype=np.float32); M[node]=D[TOP].values; Vd=sp.diags(vis_all)
for name, Ad in [('pin',(A.T@Vd).tocsr()),('pout',(A@Vd).tocsr())]:
    c=np.asarray(Ad.sum(1)).ravel(); mean=(Ad@M)/np.maximum(c,1)[:,None]; mean[c==0]=np.nan
    mx=np.full_like(M,np.nan)
    for i in np.where(c>0)[0]: mx[i]=M[Ad.indices[Ad.indptr[i]:Ad.indptr[i+1]]].max(0)
    mean=mean[node]; mx=mx[node]
    for j,f in enumerate(TOP): G[f'{name}_mean_{f}']=mean[:,j]
    for j,f in enumerate(TOP): G[f'{name}_max_{f}']=mx[:,j]
    for j,f in enumerate(TOP): G[f'{name}_diff_{f}']=D[f].values-mean[:,j]
Au=((A+A.T)>0).astype(float); Au2=Au@Au; Au2.setdiag(0); Au2=(Au2>0).astype(float)@Vd
c2=np.asarray(Au2.sum(1)).ravel(); m2=(Au2@M)/np.maximum(c2,1)[:,None]; m2[c2==0]=np.nan
for j,f in enumerate(TOP): G[f'h2_mean_{f}']=m2[node][:,j]
G['h2_cnt']=c2[node]
G = pd.DataFrame(G, index=D.index); D = pd.concat([D, G], axis=1)
GR = list(G.columns); COLS = RAW + GR
print('features:', len(COLS))

features: 388


## 4. Collective-classification machinery
- Visible graph = labelled train nodes plus test nodes.
- `nbp` builds the rich neighbour-prediction features. `nbx` adds directed 2-hop and diffusion features.
- `stage` returns out-of-fold predictions (GroupKFold over time steps) for labelled nodes and full-fit predictions for test.
- `stage_perm(rs)` is the bagged variant, with a random time-step → fold map.

In [7]:
n = len(D); y = D.label.fillna(0).values
Lm = D.label.notna().values; T = (D.is_test==1).values; ts = D.time_step.values
pos = pd.Series(np.arange(n), index=D.txId)
ee = edges[edges.txId1.isin(pos.index) & edges.txId2.isin(pos.index)]
A = sp.csr_matrix((np.ones(len(ee)), (pos[ee.txId1].values, pos[ee.txId2].values)), shape=(n,n))
V = sp.diags((Lm|T).astype(float))
Ain = (A.T@V).tocsr(); Aout = (A@V).tocsr(); U = ((A+A.T)>0).astype(float)
U2 = U@U; U2.setdiag(0); U2.eliminate_zeros(); U2 = (((U2>0).astype(float)-U.multiply(U2>0))>0).astype(float); U2 = (U2@V).tocsr()
Aii = (Ain@Ain).tocsr(); Aoo = (Aout@Aout).tocsr()
Uv = (U@V).tocsr(); dv = np.asarray(Uv.sum(1)).ravel(); P = sp.diags(1/np.maximum(dv,1))@Uv

def _max(M, p):
    c = np.asarray(M.sum(1)).ravel(); mx = np.full(n, np.nan); d = p[M.indices]
    for i in np.where(c>0)[0]: mx[i] = d[M.indptr[i]:M.indptr[i+1]].max()
    return c, mx
def nbp(p):
    out = {}
    for M,nm in [(Ain,'in'),(Aout,'out'),(U2,'h2')]:
        c,mx = _max(M,p); s = M@p
        out[f'np_{nm}_mean'] = np.where(c>0, s/np.maximum(c,1), np.nan); out[f'np_{nm}_sum'] = s; out[f'np_{nm}_max'] = mx
        out[f'np_{nm}_cnt50'] = M@(p>0.5).astype(float); out[f'np_{nm}_cnt20'] = M@(p>0.2).astype(float)
    return pd.DataFrame(out, index=D.index)
def nbx(p):
    out = nbp(p)
    for M,nm in [(Aii,'ii'),(Aoo,'oo')]:
        c,mx = _max(M,p); out[f'np_{nm}_mean'] = np.where(c>0,(M@p)/np.maximum(c,1),np.nan); out[f'np_{nm}_max'] = mx
    for a in (0.5,0.8):
        q = p.copy()
        for _ in range(10): q = (1-a)*p + a*(P@q)
        out[f'np_prop{int(a*10)}'] = np.where(dv>0,q,np.nan); out[f'np_prop{int(a*10)}_diff'] = q-p
    return out
def LG(seed): return lgb.LGBMClassifier(n_estimators=N_EST, learning_rate=0.03, num_leaves=31, colsample_bytree=0.5,
                                        subsample=0.8, subsample_freq=1, verbose=-1, n_jobs=2, random_state=seed)
def CB(seed): return cb.CatBoostClassifier(iterations=CAT_ITERS, learning_rate=0.05, depth=6, rsm=0.5, verbose=0,
                                           thread_count=2, random_seed=seed)
XGB_PARAMS = dict(n_estimators=1200, learning_rate=0.0219820953817024, max_depth=8, min_child_weight=2.9509820785603815,
                  colsample_bytree=0.5044787703803774, subsample=0.8343058365851261, reg_lambda=0.040551357415799794,
                  gamma=4.512228827591481, scale_pos_weight=1.5843236333696984)   # Optuna (TPE) on temporal folds
def XG(seed): return xgb.XGBClassifier(**{**XGB_PARAMS, 'n_estimators': 100 if SMOKE else 1200}, tree_method='hist', n_jobs=2, random_state=seed)
def stage(X, model, seeds):
    p = np.zeros(n); p[T] = np.mean([model(s).fit(X[Lm],y[Lm]).predict_proba(X[T])[:,1] for s in seeds],0)
    Ld = np.where(Lm)[0]
    for tri,vai in GroupKFold(5).split(Ld, groups=ts[Ld]):
        p[Ld[vai]] = model(0).fit(X[Ld[tri]],y[Ld[tri]]).predict_proba(X[Ld[vai]])[:,1]
    return p
def stage_perm(rs):
    def st(X, model, seeds):
        p = np.zeros(n); p[T] = np.mean([model(s+rs).fit(X[Lm],y[Lm]).predict_proba(X[T])[:,1] for s in seeds],0)
        g = np.random.RandomState(rs).permutation(36)%5; Ld = np.where(Lm)[0]; f = g[ts[Ld]]
        for k in range(5):
            tr,va = Ld[f!=k], Ld[f==k]; p[va] = model(rs).fit(X[tr],y[tr]).predict_proba(X[va])[:,1]
        return p
    return st
def pipeline(runs, feat=nbp, final=LG, seeds=SEEDS, pseudo=None):
    """2-round collective classification, bagged over `runs`. Returns test scores (rank-averaged if bagged).
    pseudo=(yp, trm): optional pseudo-labels for the final stage (labels yp on training mask trm)."""
    Xb = D[COLS]; outs = []
    yf, trf = (y, Lm) if pseudo is None else pseudo
    for rs in runs:
        st = stage if rs==0 else stage_perm(rs)
        p = st(Xb.values, LG, [0,1])                                       # stage 1
        X = pd.concat([Xb, feat(p)], axis=1).values; p = st(X, LG, [0,1])  # stage 2
        X = pd.concat([Xb, feat(p)], axis=1).values                        # final stage
        outs.append(np.mean([final(s+rs).fit(X[trf],yf[trf]).predict_proba(X[T])[:,1] for s in seeds],0))
        print('  run', rs, 'done', flush=True)
    return np.mean([rankdata(o)/len(o) for o in outs],0) if len(outs)>1 else outs[0]

## 5. Validation note
Offline temporal folds saturate at AUC ≈ 0.997–0.999 and could not separate these variants. The previous notebook version shows collective classification beating plain LightGBM on the recent windows (0.9977 vs 0.9964 and 0.9988 vs 0.9986).

Component choices and blend weights were therefore chosen on the public leaderboard. Each step was a single, interpretable change, which limits overfitting to the 30% public split. The final private score is computed on the other 70%.

## 6. Teacher: train the four components on all labelled data (ts 1–35), predict the test set (ts 36–49), rank-blend them

In [8]:
print('A: stack2_rich x', len(BAGS_A));           pA = pipeline(BAGS_A)
print('B: stack2_x x', len(BAGS_B));              pB = pipeline(BAGS_B, feat=nbx)
print('C: stack2_rich + CatBoost x', len(BAGS_B)); pC = pipeline(BAGS_B, final=CB, seeds=CAT_SEEDS)
print('D: stack2_rich + XGBoost x', len(BAGS_B));  pD = pipeline(BAGS_B, final=XG, seeds=CAT_SEEDS)
r = lambda v: pd.Series(v).rank(pct=True).values
rA,rB,rC,rD = r(pA),r(pB),r(pC),r(pD)
final       = 0.20*rA + 0.30*rB + 0.25*rC + 0.25*rD     # blend4_x30  (public 0.97039)
final_equal = 0.25*(rA + rB + rC + rD)                  # blend4_equal (public 0.97033)
print('rank correlations vs A:', {k: np.corrcoef(rA,v)[0,1].round(3) for k,v in zip('BCD',(rB,rC,rD))})

A: stack2_rich x 5
  run 0 done
  run 11 done
  run 22 done
  run 33 done
  run 44 done
B: stack2_x x 3
  run 0 done
  run 11 done
  run 22 done
C: stack2_rich + CatBoost x 3
  run 0 done
  run 11 done
  run 22 done
D: stack2_rich + XGBoost x 3
  run 0 done
  run 11 done
  run 22 done
rank correlations vs A: {'B': np.float64(0.959), 'C': np.float64(0.916), 'D': np.float64(0.96)}


## 7. Transductive pseudo-labelling (final model, `stack2_pseudo_f_bag3`, public 0.97462)
The test period (ts 36–49) differs from training: illicit behaviour shifts after the dark-market shutdown around ts 43. We use the 4-way blend above as a **teacher**. Test transactions it ranks in the **top 9%** are labelled illicit, and those in the **bottom 85%** are labelled licit. The final stage of the 2-round collective pipeline is then retrained on the real labels plus these pseudo-labels, and bagged over 3 runs.

No true test labels are used. The model only learns from its own confident predictions on the unlabelled test period: a form of semi-supervised domain adaptation.

In [ ]:
QP, QN = 0.09, 0.85
teach = pd.Series(final).rank(pct=True).values           # teacher scores on test rows (blend4_x30)
tq = np.zeros(n); tq[T] = teach
pos = T & (tq > 1-QP); neg = T & (tq < QN)
yp = y.copy(); yp[pos] = 1; yp[neg] = 0; trm = Lm | pos | neg
print('pseudo-labels: illicit', int(pos.sum()), '| licit', int(neg.sum()))
pF = pipeline(BAGS_B, pseudo=(yp, trm))                   # stack2_pseudo_f_bag3

In [ ]:
sub = pd.DataFrame({'index': D.test_index[T].values.astype(int), 'target': pF}).sort_values('index')
assert len(sub)==len(te) and sub['index'].tolist()==list(range(len(te)))
sub.to_csv('submission_final.csv', index=False)          # stack2_pseudo_f_bag3 (final pick #1)
pd.DataFrame({'index': D.test_index[T].values.astype(int), 'target': final}).sort_values('index').to_csv('submission_teacher_blend4_x30.csv', index=False)
sub.describe()